In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from loguru import logger

from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, auc, roc_curve, roc_auc_score

import dagshub
import mlflow
import shap
from mlflow.models import infer_signature

import optuna

pd.set_option('display.max_columns', None)
pd.options.display.max_info_columns = 200
pd.set_option('display.float_format', '{:.4f}'.format)


dagshub.init(repo_owner='kerasPro', repo_name='credit-risk-model', mlflow=True)

Initialized MLflow to track repo "kerasPro/credit-risk-model"

Repository kerasPro/credit-risk-model initialized!

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_parquet("../data/processed/03_feature_engineering.parquet")

df.head()

,loan_amnt,term,int_rate,sub_grade,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,dti,delinq_2yrs,fico_range_low,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,collections_12_mths_ex_med,mths_since_last_major_derog,application_type,acc_now_delinq,tot_coll_amt,tot_cur_bal,mths_since_rcnt_il,total_rev_hi_lim,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,disbursement_method,never_mths_since_last_record,never_mths_since_recent_bc_dlq,never_mths_since_last_major_derog,never_mths_since_recent_revol_delinq,never_mths_since_rcnt_il,never_mths_since_last_delinq,never_mths_since_recent_inq,antiguedad_crediticia_meses
0,3600.0000,36 months,13.9900,C4,10+ years,MORTGAGE,55000.0000,Not Verified,0,debt_consolidation,5.9100,0.0000,675.0000,1.0000,30.0000,999.0000,7.0000,0.0000,2765.0000,29.7000,13.0000,w,0.0000,30.0000,Individual,0.0000,722.0000,144904.0000,21.0000,9300.0000,4.0000,20701.0000,1506.0000,37.2000,0.0000,0.0000,148.0000,128.0000,3.0000,3.0000,1.0000,4.0000,69.0000,4.0000,69.0000,2.0000,2.0000,4.0000,2.0000,5.0000,3.0000,4.0000,9.0000,0.0000,0.0000,0.0000,3.0000,76.9000,0.0000,0.0000,0.0000,178050.0000,7746.0000,2400.0000,13734.0000,Cash,1,0,0,0,0,0,0,147.9961
1,24700.0000,36 months,11.9900,C1,10+ years,MORTGAGE,65000.0000,Not Verified,0,small_business,16.0600,1.0000,715.0000,4.0000,6.0000,999.0000,22.0000,0.0000,21470.0000,19.2000,38.0000,w,0.0000,999.0000,Individual,0.0000,0.0000,204396.0000,19.0000,111800.0000,4.0000,9733.0000,57830.0000,27.1000,0.0000,0.0000,113.0000,192.0000,2.0000,2.0000,4.0000,2.0000,999.0000,0.0000,6.0000,0.0000,5.0000,5.0000,13.0000,17.0000,6.0000,20.0000,27.0000,0.0000,0.0000,0.0000,2.0000,97.4000,7.7000,0.0000,0.0000,314017.0000,39475.0000,79300.0000,24667.0000,Cash,1,1,1,0,0,0,0,191.9842
2,20000.0000,60 months,10.7800,B4,10+ years,MORTGAGE,63000.0000,Not Verified,0,home_improvement,10.7800,0.0000,695.0000,0.0000,999.0000,999.0000,6.0000,0.0000,7869.0000,56.2000,18.0000,w,0.0000,999.0000,Joint App,0.0000,0.0000,189699.0000,19.0000,14000.0000,6.0000,31617.0000,2737.0000,55.9000,0.0000,0.0000,125.0000,184.0000,14.0000,14.0000,5.0000,101.0000,999.0000,10.0000,999.0000,0.0000,2.0000,3.0000,2.0000,4.0000,6.0000,4.0000,7.0000,0.0000,0.0000,0.0000,0.0000,100.0000,50.0000,0.0000,0.0000,218418.0000,18696.0000,6200.0000,14877.0000,Cash,1,1,1,1,0,1,0,183.9685
3,10400.0000,60 months,22.4500,F1,3 years,MORTGAGE,104433.0000,Source Verified,0,major_purchase,25.3700,1.0000,695.0000,3.0000,12.0000,999.0000,12.0000,0.0000,21929.0000,64.5000,35.0000,w,0.0000,999.0000,Individual,0.0000,0.0000,331730.0000,14.0000,34000.0000,10.0000,27644.0000,4567.0000,77.5000,0.0000,0.0000,128.0000,210.0000,4.0000,4.0000,6.0000,4.0000,12.0000,1.0000,12.0000,0.0000,4.0000,6.0000,5.0000,9.0000,10.0000,7.0000,19.0000,0.0000,0.0000,0.0000,4.0000,96.6000,60.0000,0.0000,0.0000,439570.0000,95768.0000,20300.0000,88097.0000,Cash,1,0,1,0,0,0,0,209.9869
4,11950.0000,36 months,13.4400,C3,4 years,RENT,34000.0000,Source Verified,0,debt_consolidation,10.2000,0.0000,690.0000,0.0000,999.0000,999.0000,5.0000,0.0000,8822.0000,68.4000,6.0000,w,0.0000,999.0000,Individual,0.0000,0.0000,12798.0000,338.0000,12900.0000,0.0000,2560.0000,844.0000,91.0000,0.0000,0.0000,338.0000,54.0000,32.0000,32.0000,0.0000,36.0000,999.0000,999.0000,999.0000,0.0000,2.0000,3.0000,2.0000,2.0000,2.0000,4.0000,4.0000,0.0000,0.0000,0.0

Variables con porcentaje menor a 15%, son variables imputables

In [4]:
(df.isnull().sum()*100/len(df)).sort_values(ascending= False).head(15)

num_tl_120dpd_2m             8.8722
mo_sin_old_il_acct           7.9667
emp_length                   5.8705
pct_tl_nvr_dlq               5.1425
avg_cur_bal                  5.1329
mo_sin_rcnt_rev_tl_op        5.1313
num_rev_accts                5.1313
mo_sin_old_rev_tl_op         5.1313
tot_cur_bal                  5.1313
num_actv_bc_tl               5.1313
num_tl_op_past_12m           5.1313
num_tl_90g_dpd_24m           5.1313
tot_hi_cred_lim              5.1313
total_il_high_credit_limit   5.1313
num_tl_30dpd                 5.1313
dtype: float64

In [5]:
numeric_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_features.remove('loan_status')
categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()

In [6]:
print("Numericos: ", numeric_features)
print("Categoricos: ", categorical_features)

Numericos:  ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'delinq_2yrs', 'fico_range_low', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'mths_since_rcnt_il', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 'mths_since_recent_inq', 'mths_since_recent_revol_delinq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies', 'tax_liens', 'tot_hi_

In [7]:
for col in categorical_features:
    print(f"Columna: {col} - Valores únicos: {df[col].nunique()}")
    print(df[col].value_counts())

Columna: term - Valores únicos: 2
term
36 months    1035680
60 months     333886
Name: count, dtype: int64
Columna: sub_grade - Valores únicos: 35
sub_grade
C1    86975
B4    84376
B5    83961
B3    82769
C2    80732
C3    76605
C4    76244
B2    74846
B1    71938
C5    69321
A5    64468
A4    52588
D1    52534
D2    45943
A1    43827
D3    40480
A3    38255
A2    37377
D4    36586
D5    30875
E1    24311
E2    21924
E3    18967
E4    16183
E5    14973
F1    10231
F2     7374
F3     6260
F4     5002
F5     4079
G1     3116
G2     2200
G3     1677
G4     1361
G5     1208
Name: count, dtype: int64
Columna: emp_length - Valores únicos: 11
emp_length
10+ years    449252
2 years      124063
< 1 year     110566
3 years      109662
1 year        90285
5 years       85645
4 years       82176
6 years       63798
8 years       61566
7 years       60495
9 years       51657
Name: count, dtype: int64
Columna: home_ownership - Valores únicos: 6
home_ownership
MORTGAGE    676160
RENT        545154
OW

In [8]:
X, y = df.drop("loan_status", axis=1), df["loan_status"]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(1095652, 73) (273914, 73) (1095652,) (273914,)


## Transformacion de datos para model

In [11]:
categorical_features = ['home_ownership', 'verification_status', 'purpose', 'term', 'initial_list_status', 'application_type', 'disbursement_method']

ordinal_features = ['sub_grade', 'emp_length']

# Mapeo para 'sub_grade' (de peor a mejor)
sub_grade_map = [
    'G5', 'G4', 'G3', 'G2', 'G1',
    'F5', 'F4', 'F3', 'F2', 'F1',
    'E5', 'E4', 'E3', 'E2', 'E1',
    'D5', 'D4', 'D3', 'D2', 'D1',
    'C5', 'C4', 'C3', 'C2', 'C1',
    'B5', 'B4', 'B3', 'B2', 'B1',
    'A5', 'A4', 'A3', 'A2', 'A1'
]

# Mapeo para 'emp_length' (de menos a más antigüedad)
# Importante: El SimpleImputer rellenará los NaN, que luego se tratarán como desconocidos.


emp_length_map = [
    '< 1 year', 
    '1 year', 
    '2 years', 
    '3 years', 
    '4 years', 
    '5 years', 
    '6 years', 
    '7 years', 
    '8 years', 
    '9 years', 
    '10+ years'
]

# Creamos la lista de mapeos en el orden de 'ordinal_features'
ordinal_mappings = [sub_grade_map, emp_length_map]

In [12]:
# Pipeline para Categóricas de Baja Cardinalidad
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Pipeline para Categóricas Ordinales
ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # Le pasamos los mapeos para asegurar el orden correcto
    ('ordinal', OrdinalEncoder(
        categories=ordinal_mappings,
        handle_unknown='use_encoded_value', 
        unknown_value=-1 # Asigna -1 a cualquier NaN o valor no esperado
    ))
])

# --- 4. Crear el "Jefe de Taller" (ColumnTransformer) ---
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features), # Arboles manejan NaNs y escala
        ("cat", categorical_transformer, categorical_features),
        ("ord", ordinal_transformer, ordinal_features) # <-- 'emp_length' ahora se procesa aquí
    ],
)


In [13]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_processed= pd.DataFrame(X_train_processed, columns=feature_names)
X_test_processed  = pd.DataFrame(X_test_processed,  columns=feature_names)

In [14]:
ratio = y_train.value_counts()[0] / y_train.value_counts()[1]
model = LGBMClassifier(scale_pos_weight=ratio, random_state=42, n_jobs=8)

In [15]:
def calculate_classification_metrics(y_true, y_pred) -> dict:
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': auc,
    }

In [16]:
model.fit(X_train_processed, y_train)
y_pred = model.predict(X_test_processed)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 232661, number of negative: 862991
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.059190 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7738
[LightGBM] [Info] Number of data points in the train set: 1095652, number of used features: 97
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.212349 -> initscore=-1.310822
[LightGBM] [Info] Start training from score -1.310822


In [17]:
calculate_classification_metrics(y_test, y_pred)

{'accuracy': 0.6571369115853881,
 'precision': 0.3456899900720853,
 'recall': 0.688426228380841,
 'f1_score': 0.46026172263377796,
 'roc_auc': 0.6685637454824835}

## FEATURE SELECTION

In [18]:
feature_names = preprocessor.get_feature_names_out()

importances = model.feature_importances_

feature_importances = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

feature_importances.head(10)


,feature,importance
0,num__loan_amnt,273
3,num__dti,136
2,num__annual_inc,131
1,num__int_rate,124
96,ord__emp_length,112
28,num__mo_sin_old_rev_tl_op,105
21,num__acc_open_past_24mths,105
87,cat__term_ 36 months,104
95,ord__sub_grade,96
38,num__num_actv_rev_tl,94


Queremos saber con cuantas variables nos podemos quedar sin perder Rendimiento (RECALL)

In [19]:
opcions = [20, 30, 40, 50, 60, 70, 80]
list_scores = []
for n in opcions:
    best_features = feature_importances["feature"].head(n).to_list()
    model.fit(X_train_processed[best_features], y_train)
    y_pred = model.predict(X_test_processed[best_features])
    scores = calculate_classification_metrics(y_test, y_pred)
    scores['n_features'] = n
    list_scores.append(scores)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 232661, number of negative: 862991
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009453 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3285
[LightGBM] [Info] Number of data points in the train set: 1095652, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.212349 -> initscore=-1.310822
[LightGBM] [Info] Start training from score -1.310822
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 232661, number of negative: 862991
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022180 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `f

In [20]:
for score in list_scores:
    print(score)

{'accuracy': 0.6550231094431099, 'precision': 0.34344888214710495, 'recall': 0.6850909466011071, 'f1_score': 0.4575296224855332, 'roc_auc': 0.6660038599368143, 'n_features': 20}
{'accuracy': 0.6564505647758055, 'precision': 0.3450336780191633, 'recall': 0.6878073101124368, 'f1_score': 0.45954157262070905, 'roc_auc': 0.6679020235231335, 'n_features': 30}
{'accuracy': 0.6575969099790445, 'precision': 0.34604247521414805, 'recall': 0.68827149881374, 'f1_score': 0.4605394088244937, 'roc_auc': 0.6687992457081103, 'n_features': 40}
{'accuracy': 0.6572975459450776, 'precision': 0.34585862423263886, 'recall': 0.6886669188185538, 'f1_score': 0.4604649826134437, 'roc_auc': 0.6687536162589349, 'n_features': 50}
{'accuracy': 0.6574435771811591, 'precision': 0.34598609491730364, 'recall': 0.6887184953409208, 'f1_score': 0.4605894763467873, 'roc_auc': 0.6688651527078189, 'n_features': 60}
{'accuracy': 0.6571734193944084, 'precision': 0.3458289835816646, 'recall': 0.689131107519857, 'f1_score': 0.460

## OPTIMIZACIÓN DE HIPERPARÁMETROS

Para ahorrar recursos computacionales en mi máquina local, se decidió usar 30 de las mejores variables.
Porque es el más equilibrado en cuanto a f1_score y recall. Nuestra principal métrica para este caso es RECALL

In [21]:
best_features = feature_importances["feature"].head(30).to_list()
X_train_final = X_train_processed[best_features]
X_test_final  = X_test_processed[best_features]

ratio = y_train.value_counts()[0] / y_train.value_counts()[1]


In [35]:
y_test

100416     1
26853      0
1267078    1
1068483    1
1324864    0
          ..
1174190    0
946578     0
201479     0
926964     1
427081     0
Name: loan_status, Length: 273914, dtype: int64

In [36]:
#GUARDAR DATA

testing_data = X_test_final.copy()
testing_data["loan_status"] = y_test.values

testing_data.to_parquet("../data/processed/04_modeling_testing_data.parquet", index=False)

In [41]:
training_data = X_train_final.copy()
training_data["loan_status"] = y_train.values
training_data.to_parquet("../data/processed/04_modeling_training_data.parquet", index=False)

In [ ]:
def optuna_objective(trial):
    # Log cuando inicia un trial
    logger.info(f"=== Iniciando trial {trial.number + 1} ===")

    params = {
        "objective": "binary",
        "boosting_type": "gbdt",
        "scale_pos_weight": ratio,

        "n_estimators": trial.suggest_int("n_estimators", 80, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 127),
        "min_child_samples": trial.suggest_int("min_child_samples", 100, 600),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 3.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 3.0, log=True),

        "random_state": 100,
        "n_jobs": -1,
    }

    classifier = LGBMClassifier(**params)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=100)
    scores = cross_val_score(
        classifier,
        X_train_final,
        y_train,
        cv=cv,
        scoring='recall',
        n_jobs= 1
    )
    _recall = np.mean(scores)

    # Log cuando termina un trial
    logger.info(f"=== Trial {trial.number + 1} finalizado | Recall: {_recall:.4f} ===")

    return 1 - _recall


with mlflow.start_run(run_name="Optuna_TPE_2"):

    logger.info("===== Iniciando estudio Optuna =====")
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42)
    )

    study.optimize(optuna_objective, n_trials=30, n_jobs=1)

    logger.info("===== Estudio Optuna completado =====")
    logger.info(f"Mejor valor (1 - recall): {study.best_value}")
    logger.info(f"Mejores parámetros: {study.best_params}")

    optuna_best_params = study.best_params

    # Entrenar el modelo final con los mejores parámetros
    best_classifier = LGBMClassifier(**optuna_best_params,objective="binary", boosting_type="gbdt",
                                    scale_pos_weight= ratio,)
    
    best_classifier.fit(X_train_final, y_train)

    y_pred_optuna = best_classifier.predict(X_test_final)
    classification_metrics_optuna = calculate_classification_metrics(y_test, y_pred_optuna)
    logger.info(f"Classification metrics after Optuna TPE: {classification_metrics_optuna}")
    
    mlflow.sklearn.log_model(
        sk_model=best_classifier, 
        artifact_path="lightgbm_classifier_model",
        input_example=X_train_final.head(),
        signature=infer_signature(X_train_final, best_classifier.predict(X_train_final))
    )

    mlflow.log_params(optuna_best_params)

    for metric_name, metric_value in classification_metrics_optuna.items():
        mlflow.log_metric(metric_name, metric_value)



In [31]:
with mlflow.start_run(run_name="XGBoost_Model"):
    xgb_model = XGBClassifier(
        objective="binary:logistic",
        scale_pos_weight=ratio,
        random_state=42,
        n_jobs=8
    )

    xgb_model.fit(X_train_final, y_train)

    y_pred_xgb = xgb_model.predict(X_test_final)
    classification_metrics_xgb = calculate_classification_metrics(y_test, y_pred_xgb)
    logger.info(f"Classification metrics XGBoost: {classification_metrics_xgb}")

    mlflow.sklearn.log_model(
        sk_model=xgb_model, 
        artifact_path="xgboost_classifier_model",
        input_example=X_train_final.head(),
        signature=infer_signature(X_train_final, xgb_model.predict(X_train_final))
    )

    for metric_name, metric_value in classification_metrics_xgb.items():
        mlflow.log_metric(metric_name, metric_value)

2025-11-16 09:15:10.867 | INFO     | __main__:<module>:13 - Classification metrics XGBoost: {'accuracy': 0.6649167256876246, 'precision': 0.34977478282629676, 'recall': 0.6728501186260014, 'f1_score': 0.460278258006092, 'roc_auc': 0.6678139945522613}


In [33]:
with mlflow.start_run(run_name="Final_RandomForest_Model"):
    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42,
        n_jobs=8
    )

    rf_model.fit(X_train_final, y_train)

    y_pred_rf = rf_model.predict(X_test_final)
    classification_metrics_rf = calculate_classification_metrics(y_test, y_pred_rf)
    logger.info(f"Classification metrics Random Forest: {classification_metrics_rf}")

    mlflow.sklearn.log_model(
        sk_model=rf_model, 
        artifact_path="random_forest_classifier_model",
        input_example=X_train_final.head(),
        signature=infer_signature(X_train_final, rf_model.predict(X_train_final))
    )

    for metric_name, metric_value in classification_metrics_rf.items():
        mlflow.log_metric(metric_name, metric_value)

2025-11-16 09:18:18.631 | INFO     | __main__:<module>:16 - Classification metrics Random Forest: {'accuracy': 0.6501164599107749, 'precision': 0.3380952380952381, 'recall': 0.6762369769281024, 'f1_score': 0.45080398381717535, 'roc_auc': 0.6596556521921042}


### Crear el mejor modelo

In [40]:
best_classifier = LGBMClassifier(**optuna_best_params,objective="binary", boosting_type="gbdt",
                                    scale_pos_weight= ratio)

best_classifier.fit(X_train_final, y_train)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 232661, number of negative: 862991
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.043170 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4931
[LightGBM] [Info] Number of data points in the train set: 1095652, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.212349 -> initscore=-1.310822
[LightGBM] [Info] Start training from score -1.310822


,boosting_type,'gbdt'
,num_leaves,34
,max_depth,-1
,learning_rate,0.06656333325146147
,n_estimators,106
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,555


In [42]:
# Guardar best model

import joblib

joblib.dump(best_classifier, "../models/modelo_credito.pkl")

['../models/modelo_credito.pkl']